In [12]:
import pandas as pd
from datasets import Dataset, DatasetDict
import os

def load_csv_data(csv_path):
    """Load CSV file with two columns"""
    df = pd.read_csv(csv_path)

    # Ensure we only have two columns
    if len(df.columns) != 2:
        raise ValueError(f"CSV file should have exactly 2 columns, but found {len(df.columns)}")

    # Get column names
    col1, col2 = df.columns.tolist()
    print(f"CSV columns: {col1}, {col2}")

    return df

def load_txt_data(txt_path, col1_name="text", col2_name="label"):
    """Load TXT file and create a DataFrame with specified column names"""
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Remove trailing newlines
    lines = [line.strip() for line in lines if line.strip()]

    # Create DataFrame - assuming each line is text with a default label
    # You can modify this based on your TXT file format
    data = []
    for line in lines:
        # If TXT contains tab-separated values, split them
        if '\t' in line:
            parts = line.split('\t', 1)  # Split on first tab only
            if len(parts) == 2:
                data.append({col1_name: parts[0], col2_name: parts[1]})
            else:
                data.append({col1_name: line, col2_name: ""})
        else:
            # If no tabs, treat entire line as text with empty label
            data.append({col1_name: line, col2_name: ""})

    return pd.DataFrame(data)

def create_text_generation_dataset(csv_path, txt_path, output_path=None):
    """
    Create HuggingFace dataset from CSV and TXT files with SINGLE TEXT COLUMN for text generation

    Args:
        csv_path: Path to CSV file with two columns
        txt_path: Path to TXT file
        output_path: Optional path to save the dataset
    """

    print(f"Loading CSV from: {csv_path}")
    print(f"Loading TXT from: {txt_path}")

    # Check if files exist
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    if not os.path.exists(txt_path):
        raise FileNotFoundError(f"TXT file not found: {txt_path}")

    # Load CSV data
    csv_df = load_csv_data(csv_path)
    csv_columns = csv_df.columns.tolist()

    # Load TXT data with same column names as CSV
    txt_df = load_txt_data(txt_path, csv_columns[0], csv_columns[1])

    # Concatenate the dataframes
    combined_df = pd.concat([csv_df, txt_df], ignore_index=True)

    # Remove any rows with NaN values
    combined_df = combined_df.dropna()

    print(f"CSV rows: {len(csv_df)}")
    print(f"TXT rows: {len(txt_df)}")
    print(f"Total rows before processing: {len(combined_df)}")

    # CREATE SINGLE TEXT COLUMN FOR TEXT GENERATION
    print("Creating single text column for text generation...")

    # Combine both columns into one text field
    text_data = []
    for _, row in combined_df.iterrows():
        # Format: User: {user_message}\nAssistant: {response}
        combined_text = f"User: {row[csv_columns[0]]}\nAssistant: {row[csv_columns[1]]}"
        text_data.append({"text": combined_text})

    # Create new DataFrame with only text column
    text_df = pd.DataFrame(text_data)

    print(f"Final dataset shape: {text_df.shape}")
    print(f"Final columns: {list(text_df.columns)}")
    print(f"Total text examples: {len(text_df)}")

    # Show sample of combined text
    print(f"\nSample combined text (first 300 chars):")
    print(f"'{text_df['text'].iloc[0][:300]}...'")

    # Create HuggingFace Dataset
    dataset = Dataset.from_pandas(text_df)

    # Wrap in DatasetDict with only train split
    dataset_dict = DatasetDict({
        'train': dataset
    })

    print(f"\nFinal HuggingFace Dataset Info:")
    print(dataset_dict)
    print(f"Features: {list(dataset_dict['train'].features.keys())}")
    print(f"Train split size: {len(dataset_dict['train'])}")

    # Save dataset if output path is provided
    if output_path:
        dataset_dict.save_to_disk(output_path)
        print(f"Dataset saved to: {output_path}")

    return dataset_dict

# ============= CREATE TEXT GENERATION DATASET =============

# Use your specific file paths
csv_file = "/content/telegram_pairs.csv"
txt_file = "/content/chat.txt"

# Create the dataset for text generation
print("Creating TEXT GENERATION dataset with SINGLE TEXT COLUMN...")
print("="*60)
dataset = create_text_generation_dataset(csv_file, txt_file)

# Show sample data
print("\n" + "="*60)
print("SAMPLE DATA FROM TRAIN SPLIT:")
print("="*60)

for i in range(min(3, len(dataset['train']))):
    example = dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"{example['text'][:400]}...")  # Show first 400 characters
    print("-" * 50)

print(f"\n" + "="*60)
print(f"TEXT GENERATION DATASET READY!")
print(f"Total examples: {len(dataset['train'])}")
print(f"Single column: 'text'")
print(f"Perfect for text generation training!")
print("="*60)

# Access your dataset:
# train_data = dataset['train']
# first_example = dataset['train'][0]['text']
# print(first_example)

# Save if needed:
# dataset.save_to_disk("/content/text_generation_dataset")

Creating TEXT GENERATION dataset with SINGLE TEXT COLUMN...
Loading CSV from: /content/telegram_pairs.csv
Loading TXT from: /content/chat.txt
CSV columns: user_message, response
CSV rows: 23570
TXT rows: 27692
Total rows before processing: 51262
Creating single text column for text generation...
Final dataset shape: (51262, 1)
Final columns: ['text']
Total text examples: 51262

Sample combined text (first 300 chars):
'User: دوره اول هوش‌مصنوعی
Assistant: ما یه گروه دیگه زدیم از قبل ۶ تا هستیم...'

Final HuggingFace Dataset Info:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 51262
    })
})
Features: ['text']
Train split size: 51262

SAMPLE DATA FROM TRAIN SPLIT:

Example 1:
User: دوره اول هوش‌مصنوعی
Assistant: ما یه گروه دیگه زدیم از قبل ۶ تا هستیم...
--------------------------------------------------

Example 2:
User: ما یه گروه دیگه زدیم از قبل ۶ تا هستیم
Assistant: میخواین اونجا سوئیچ بشیم؟...
--------------------------------------------------

Exa

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import notebook_login

In [14]:
notebook_login()

In [15]:
# check_point = "google/gemma-3-1b-it"
check_point = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(check_point)
tokenizer.add_special_tokens({
    "pad_token": "<PAD>",
    "bos_token": tokenizer.bos_token or "<BOS>",
    "eos_token": tokenizer.eos_token or "<EOS>"
})

def preproccess(example):
    tokenized = tokenizer(example["text"], truncation=True,)#padding="max_length", max_length=128)
    # tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
dataset = dataset.map(preproccess, batched=True)

Map:   0%|          | 0/51262 [00:00<?, ? examples/s]

In [16]:
model = AutoModelForCausalLM.from_pretrained(check_point)
model.resize_token_embeddings(len(tokenizer))
# train_val = dataset["train"].train_test_split(test_size=0.05)
# train_set = train_val["train"]
# dev_set = train_val["test"]
train_set = dataset["train"]

In [24]:
from transformers import TrainingArguments, Trainer
train_args = TrainingArguments(
    # eval_strategy="steps",
    report_to="none",
    learning_rate=5e-5,
    output_dir="./results",
    # per_device_eval_batch_size=1,
    per_device_train_batch_size=4,
    logging_steps=1000,
    do_train=True,
    # do_eval=True,
    # do_predict=True,
    torch_empty_cache_steps=1500,
    weight_decay=1e-6,
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    # load_best_model_at_end=True,
    greater_is_better=False,
    label_smoothing_factor=0.01,
    group_by_length=True,
    save_steps=1000,
    gradient_accumulation_steps=2,
    metric_for_best_model="perplexity"

)

In [18]:
# !pip install -q evaluate

In [25]:
import evaluate
metirc = evaluate.load("meteor")
per = evaluate.load("perplexity")
def metrics(val_data):
    pred, true = val_data
    pred = tokenizer.decode(pred, skip_special_token = True)
    true = tokenizer.decode(true, skip_special_token = True)

    meteor = metric.compute(reference=true, predictions=pred)
    per = per.compute(reference=true, predictions=pred)

    return {
        "meteor" : meteor,
        "perplexity" : per
    }


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [20]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer, mlm=False
)

In [26]:
trainer = Trainer(
    model,
    train_args,
    train_dataset=train_set,
    # eval_dataset=dev_set,
    processing_class=tokenizer,
    compute_metrics=metrics,
    data_collator=data_collator,

)

In [27]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.


Step,Training Loss
1000,4.421700
2000,3.612500
3000,3.402300
4000,3.289800
5000,3.209600
6000,3.178000


TrainOutput(global_step=6408, training_loss=3.4957717724060746, metrics={'train_runtime': 1141.1589, 'train_samples_per_second': 44.921, 'train_steps_per_second': 5.615, 'total_flos': 1625170095360000.0, 'train_loss': 3.4957717724060746, 'epoch': 1.0})

In [29]:
model.save_pretrained("finetuned-on-telegram-model")
tokenizer.save_pretrained("finetuned-on-telegram-model-tokenizer")

('finetuned-on-telegram-model-tokenizer/tokenizer_config.json',
 'finetuned-on-telegram-model-tokenizer/special_tokens_map.json',
 'finetuned-on-telegram-model-tokenizer/vocab.json',
 'finetuned-on-telegram-model-tokenizer/merges.txt',
 'finetuned-on-telegram-model-tokenizer/added_tokens.json',
 'finetuned-on-telegram-model-tokenizer/tokenizer.json')

In [31]:
from huggingface_hub import notebook_login
notebook_login()

In [32]:
model.push_to_hub("finetuned-on-telegram-model")
tokenizer.push_to_hub("finetuned-on-telegram-model-tokenizer")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...on-telegram-model/model.safetensors:   0%|          |  549kB /  498MB            

CommitInfo(commit_url='https://huggingface.co/alien8887/finetuned-on-telegram-model-tokenizer/commit/41767d15f8d792ee90f8346cf8a04aecd3014659', commit_message='Upload tokenizer', commit_description='', oid='41767d15f8d792ee90f8346cf8a04aecd3014659', pr_url=None, repo_url=RepoUrl('https://huggingface.co/alien8887/finetuned-on-telegram-model-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='alien8887/finetuned-on-telegram-model-tokenizer'), pr_revision=None, pr_num=None)

In [ ]:

model.eval()

# Chat loop
while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        break

    inputs = tokenizer(user_input, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,

    )
    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Bot:", reply)
